# Striker Analysis System

## Overview
This system performs comprehensive analysis of striker performance using event-based soccer data from StatsBomb. It evaluates strikers across multiple technical and tactical dimensions, normalizing the results to provide comparable metrics across different playing times and contexts.

## Data Collection

### Data Source
- Uses StatsBomb's event-based soccer data through the `statsbombpy` Python package.
- Collects match events filtered by competition ID and season ID.
- Primary data structure is event-based, where each row represents a discrete action on the field.

### Player Identification
- Identifies strikers through position mapping, including:
  - Center Forward
  - Striker
  - Second Striker
  - Centre Forward
  - Left/Right Center Forward
  - Forward
- Tracks individual player events across matches to build comprehensive profiles.

### Event Types Analyzed
- Shots
- Passes
- Carries
- Pressure Events
- Physical Duels
- Aerial Duels

## Performance Metrics

### 1. Finishing (40% shot quality, 30% pressure finishing, 30% technique variety)
- **Shot Quality:** Goals scored versus expected goals (xG).
- **Pressure Finishing:** Success rate of shots under defensive pressure.
- **Technique Variety:** Effectiveness across different body parts.

### 2. Early Shots
- Identifies quick shots after receiving possession.
- Measures through multiple indicators:
  - First-time shots.
  - Time since last action (≤ 2 seconds).
  - Shot technique classification.
- Default assumption: 20% of shots are quick shots if specific indicators are unavailable.

### 3. Link-up Play
- **Progressive Passes (60% weighting)**
  - Defined as passes advancing the ball ≥10 units forward.
  - Completion rate under pressure.
- **Pressure Pass Completion (40% weighting)**
  - Success rate of passes made under defensive pressure.

### 4. Carrying
- Equal weighting (50% each) between:
  - **Progressive Carries**
    - Defined as carries advancing the ball ≥10 units forward.
    - Success rate calculation.
  - **Pressure Resistance**
    - Success rate of carries under defensive pressure.
    - Measured through various pressure indicators.

### 5. Work Rate
- Equal weighting (50% each) between:
  - **Pressure Rate**
    - Normalized by team possession time.
    - Frequency of pressing actions.
  - **Counterpress Success**
    - Pressures within 5 seconds of losing possession.
    - Success rate of counterpressing actions.

### 6. Back to Goal Play
- **Physical Duels (60% weighting)**
  - Success in physical confrontations.
  - Hold-up play outcomes.
- **Hold-up Success (40% weighting)**
  - Successful actions after receiving passes under pressure.
  - Ball retention and distribution quality.

### 7. Headed Duels
- **Aerial Success (60% weighting)**
  - Success rate in aerial duels.
- **Headed Shot Accuracy (40% weighting)**
  - Accuracy of headed attempts on goal.

## Normalization Methodology

### Statistical Normalization

#### Base Calculation
- Metrics are initially calculated as raw values between 0-1.
- Each metric is averaged per match to account for varying numbers of appearances.

#### Minutes Adjustment
- Minimum threshold of 180 minutes played.
- Weight calculation: `min(max(minutes_played, 180) / 270, 1)`.
- Prevents overvaluation of small sample sizes while respecting significant playing time.

#### Final Normalization
- Percentile ranking within the player pool.
- Scaled to 0-10 range.
- Clipped to prevent extreme values.
- Rounded to 2 decimal places for readability.

## Quality Controls
- Handles missing data gracefully through fallback calculations.
- Prevents division by zero through safe division function.
- Replaces infinite values with 0.
- Returns middle value (5) when no variation exists in metric.
- Weights adjusted by playing time to prevent small sample size bias.

## Error Handling

### Robust Error Management
- Try-except blocks at multiple levels:
  - Individual metric calculations.
  - Match processing.
  - Overall analysis.
- Graceful fallbacks when specific data points are missing.
- Detailed error logging for debugging.

### Data Validation
- Checks for required columns before calculations.
- Validates data types and structures.
- Handles missing or malformed data through fallback logic.

## Output Format

### Final Dataset
- Player-level DataFrame containing:
  - All normalized metrics (0-10 scale).
  - Minutes played.
  - Matches played.
  - Average minutes per match.
- All numeric values rounded to 2 decimal places.
- Optional CSV export functionality.

## Limitations and Considerations

### Data Dependencies
- Relies on consistent position labeling in StatsBomb data.
- Some metrics require specific event attributes that may not be present in all datasets.
- Quality of analysis depends on completeness of event data.

### Statistical Considerations
- Small sample sizes are adjusted but may still affect reliability.
- Normalization assumes normal distribution of skills in player pool.
- Context (opposition quality, tactical role) not directly accounted for.


In [25]:
import pandas as pd
import numpy as np
from statsbombpy import sb
from collections import defaultdict
import sys
import glob
import os
import warnings
warnings.simplefilter("ignore")

In [26]:

def safe_divide(numerator, denominator):
    """
    Safely divide numbers, returning 0 if denominator is 0
    """
    return numerator / denominator if denominator != 0 else 0

In [27]:
def calculate_finishing(events, striker, team):
    """
    Calculate finishing metrics including shot quality, pressure finishing, and technique variety
    Returns a score between 0-1
    """
    try:
        striker_events = events[events['player'] == striker]
        
        # Get all shots
        shots = striker_events[striker_events['type'] == 'Shot']
        if len(shots) == 0:
            return 0
        
        # Shot quality (goals / xG)
        goals = len(shots[shots['shot_outcome'] == 'Goal'])
        total_xg = shots['shot_statsbomb_xg'].fillna(0).sum()
        shot_quality = safe_divide(goals, max(total_xg, 1))
        
        # Pressure finishing - use shot pressure if available, otherwise use general pressure
        if 'under_pressure' in shots.columns:
            pressure_shots = shots[shots['under_pressure'] == True]
        else:
            pressure_shots = shots[shots['pressure'] == True] if 'pressure' in shots.columns else pd.DataFrame()
        
        pressure_ratio = safe_divide(
            len(pressure_shots[pressure_shots['shot_outcome'] == 'Goal']),
            len(pressure_shots) if len(pressure_shots) > 0 else 1
        )
        
        # Technique variety (by body part)
        if 'shot_body_part' in shots.columns:
            body_parts = shots['shot_body_part'].dropna().unique()
            body_part_scores = []
            for part in body_parts:
                part_shots = shots[shots['shot_body_part'] == part]
                if len(part_shots) > 0:
                    score = safe_divide(
                        len(part_shots[part_shots['shot_outcome'] == 'Goal']),
                        len(part_shots)
                    )
                    body_part_scores.append(score)
            technique_variety = np.mean(body_part_scores) if body_part_scores else 0
        else:
            technique_variety = 0
        
        # Weight components
        return (shot_quality * 0.4) + (pressure_ratio * 0.3) + (technique_variety * 0.3)
    except Exception as e:
        print(f"Error in calculate_finishing: {str(e)}")
        return 0

In [28]:
def calculate_early_shots(events, striker, team):
    """
    Calculate early shot metrics including quick shots after receiving
    Returns a score between 0-1
    """
    try:
        striker_events = events[events['player'] == striker]
        
        # Get all shots
        shots = striker_events[striker_events['type'] == 'Shot']
        if len(shots) == 0:
            return 0
        
        # Initialize quick shots count
        quick_shots_count = 0
        
        # Use multiple indicators to identify quick shots
        if 'first_time' in shots.columns:
            quick_shots_count += len(shots[shots['first_time'] == True])
        
        if 'time_since_last_action' in shots.columns:
            quick_shots_count += len(shots[shots['time_since_last_action'].fillna(999) <= 2])
            
        # If neither column exists, estimate based on shot technique
        if 'shot_technique' in shots.columns:
            quick_shots_count += len(shots[shots['shot_technique'].isin(['First Time', 'Volley', 'Half Volley'])])
            
        # Ensure we have at least some quick shots identified
        if quick_shots_count == 0:
            quick_shots_count = len(shots) * 0.2  # Assume 20% of shots are quick shots if no other data
            
        # Calculate metrics
        quick_shot_ratio = safe_divide(quick_shots_count, len(shots))
        
        # Calculate conversion rate for quick shots
        quick_goals = len(shots[
            (shots['shot_outcome'] == 'Goal') & 
            ((shots['first_time'] == True) if 'first_time' in shots.columns else True)
        ])
        
        conversion_rate = safe_divide(quick_goals, quick_shots_count)
        
        # Weight components
        return (quick_shot_ratio * 0.5) + (conversion_rate * 0.5)
    except Exception as e:
        print(f"Error in calculate_early_shots: {str(e)}")
        return 0


In [29]:
def calculate_linkup_play(events, striker, team):
    """
    Calculate link-up play metrics including progressive passes and pass involvement
    Returns a score between 0-1
    """
    try:
        striker_events = events[events['player'] == striker]
        
        # Get all passes
        passes = striker_events[striker_events['type'] == 'Pass']
        if len(passes) == 0:
            return 0
        
        def is_progressive_pass(pass_row):
            try:
                if not (isinstance(pass_row.get('location', []), list) and 
                       isinstance(pass_row.get('pass_end_location', []), list)):
                    return False
                start = pass_row['location']
                end = pass_row['pass_end_location']
                return end[0] - start[0] >= 10
            except:
                return False
        
        # Progressive passes
        prog_passes = passes[passes.apply(is_progressive_pass, axis=1)]
        prog_completion = safe_divide(
            len(prog_passes[prog_passes['pass_outcome'].isna()]),
            len(prog_passes)
        )
        
        # Passes under pressure
        if 'under_pressure' in passes.columns:
            pressure_passes = passes[passes['under_pressure'] == True]
        else:
            pressure_passes = passes[passes['pressure'] == True] if 'pressure' in passes.columns else pd.DataFrame()
        
        pressure_completion = safe_divide(
            len(pressure_passes[pressure_passes['pass_outcome'].isna()]),
            len(pressure_passes)
        )
        
        # Weight components
        return (prog_completion * 0.6) + (pressure_completion * 0.4)
    except Exception as e:
        print(f"Error in calculate_linkup_play: {str(e)}")
        return 0


In [30]:
def calculate_carrying(events, striker, team):
    """
    Calculate carrying metrics including progressive carries and pressure resistance
    Returns a score between 0-1
    """
    try:
        striker_events = events[events['player'] == striker]
        
        # Analyze carries
        carries = striker_events[striker_events['type'] == 'Carry']
        if len(carries) == 0:
            return 0
        
        def is_progressive_carry(carry_row):
            try:
                if not (isinstance(carry_row.get('location', []), list) and 
                       isinstance(carry_row.get('carry_end_location', []), list)):
                    return False
                start = carry_row['location']
                end = carry_row['carry_end_location']
                return end[0] - start[0] >= 10
            except:
                return False
        
        # Progressive carries
        prog_carries = carries[carries.apply(is_progressive_carry, axis=1)]
        prog_carry_success = safe_divide(len(prog_carries), len(carries))
        
        # Pressure carries - check for different pressure column names
        if 'under_pressure' in carries.columns:
            pressure_carries = carries[carries['under_pressure'] == True]
        elif 'pressure' in carries.columns:
            pressure_carries = carries[carries['pressure'] == True]
        else:
            pressure_carries = pd.DataFrame()
        
        # Check for successful carries (no outcome usually means successful)
        if 'carry_end_type' in carries.columns:
            successful_pressure_carries = pressure_carries[
                ~pressure_carries['carry_end_type'].isin(['Dispossessed', 'Lost', 'Tackled'])
            ]
        else:
            # If no carry end type, consider all completed carries as successful
            successful_pressure_carries = pressure_carries
        
        pressure_carry_success = safe_divide(
            len(successful_pressure_carries),
            len(pressure_carries)
        )
        
        # Weight components
        return (prog_carry_success * 0.5) + (pressure_carry_success * 0.5)
    except Exception as e:
        print(f"Error in calculate_carrying: {str(e)}")
        return 0

In [31]:
def calculate_work_rate(events, striker, team):
    """
    Calculate work rate metrics including pressure events and counterpressing
    Returns a score between 0-1
    """
    try:
        striker_events = events[events['player'] == striker]
        
        # Get pressure events
        pressures = striker_events[striker_events['type'] == 'Pressure']
        if len(pressures) == 0:
            return 0
        
        # Calculate pressure rate (normalized by team possession)
        team_events = events[events['team'] == team]
        possession_time = max(team_events['minute'].max() - team_events['minute'].min(), 1)
        pressure_rate = len(pressures) / possession_time
        
        # Counterpress success
        if 'counterpress' in pressures.columns:
            counterpress = pressures[pressures['counterpress'] == True]
        else:
            # Estimate counterpressing as pressures within 5 seconds of losing possession
            counterpress = pressures[pressures['time_since_possession_loss'].fillna(999) <= 5] if 'time_since_possession_loss' in pressures.columns else pd.DataFrame()
        
        # Check for pressure success using different possible column names
        if 'pressure_outcome' in pressures.columns:
            successful_counterpress = counterpress[counterpress['pressure_outcome'] == 'Success']
        elif 'pressure_regain' in pressures.columns:
            successful_counterpress = counterpress[counterpress['pressure_regain'] == True]
        elif 'pressure_regain_within_5_sec' in pressures.columns:
            successful_counterpress = counterpress[counterpress['pressure_regain_within_5_sec'] == True]
        else:
            # If no success indicator is available, estimate based on ball recovery events
            ball_recoveries = events[
                (events['type'] == 'Ball Recovery') & 
                (events['player'] == striker)
            ]
            successful_counterpress = ball_recoveries
        
        counterpress_success = safe_divide(
            len(successful_counterpress),
            len(counterpress)
        )
        
        # Weight components
        return (pressure_rate * 0.5) + (counterpress_success * 0.5)
    except Exception as e:
        print(f"Error in calculate_work_rate: {str(e)}")
        return 0

In [32]:
def calculate_back_to_goal(events, striker, team):
    """
    Calculate back to goal play metrics including physical duels and hold-up play
    Returns a score between 0-1
    """
    try:
        striker_events = events[events['player'] == striker]
        
        # Initialize scores
        physical_score = 0
        holdup_score = 0
        
        # Physical duels
        duels = striker_events[striker_events['type'] == 'Duel']
        physical_duels = duels[
            (duels['duel_type'] == 'Physical') | 
            (duels['duel_type'] == 'Hold-up') |
            (duels['duel_type'] == 'Shield')
        ]
        
        if len(physical_duels) > 0:
            physical_score = safe_divide(
                len(physical_duels[physical_duels['duel_outcome'] == 'Won']),
                len(physical_duels)
            )
        
        # Hold-up play indicators
        received_passes = events[
            (events['type'] == 'Pass') & 
            (events['pass_recipient'] == striker)
        ]
        
        if len(received_passes) > 0:
            # Calculate successful hold-up play based on what happens after receiving
            successful_holdup = striker_events[
                (striker_events['type'].isin(['Pass', 'Shot'])) &
                (striker_events['under_pressure'] == True if 'under_pressure' in striker_events.columns else True)
            ]
            
            holdup_score = safe_divide(len(successful_holdup), len(received_passes))
        
        # If no physical duels, rely more on hold-up play
        if physical_score == 0:
            return holdup_score
        elif holdup_score == 0:
            return physical_score
        else:
            return (physical_score * 0.6) + (holdup_score * 0.4)
            
    except Exception as e:
        print(f"Error in calculate_back_to_goal: {str(e)}")
        return 0


In [33]:
def calculate_headed_duels(events, striker, team):
    """
    Calculate headed duel metrics including aerial success and headed shots
    Returns a score between 0-1
    """
    try:
        striker_events = events[events['player'] == striker]
        
        # Aerial duels
        aerial_duels = striker_events[
            (striker_events['type'] == 'Duel') &
            (striker_events['duel_type'] == 'Aerial')
        ]
        
        aerial_success = safe_divide(
            len(aerial_duels[aerial_duels['duel_outcome'] == 'Won']),
            len(aerial_duels)
        )
        
        # Headed shots
        shots = striker_events[striker_events['type'] == 'Shot']
        if 'shot_body_part' in shots.columns:
            headed_shots = shots[shots['shot_body_part'] == 'Head']
        else:
            headed_shots = pd.DataFrame()
        
        headed_accuracy = safe_divide(
            len(headed_shots[headed_shots['shot_outcome'].isin(['Goal', 'Saved'])]),
            len(headed_shots)
        )
        
        # Weight components
        return (aerial_success * 0.6) + (headed_accuracy * 0.4)
    except Exception as e:
        print(f"Error in calculate_headed_duels: {str(e)}")
        return 0


In [34]:
def normalize_metric(series, min_val=0, max_val=10, minutes_played=None):
    """
    Normalize metric with minutes adjustment
    """
    try:
        if len(series) == 0:
            return pd.Series([])
        
        # Replace inf and -inf with 0
        series = series.replace([np.inf, -np.inf], 0)
        
        # Fill NaN with 0
        series = series.fillna(0)
        
        if minutes_played is not None:
            min_minutes = 180
            weights = np.minimum(np.maximum(minutes_played, min_minutes) / 270, 1)
            series = series * weights
        
        if series.std() == 0:
            return pd.Series([5] * len(series))  # Return middle value if no variation
        
        percentiles = series.rank(pct=True)
        normalized = percentiles * max_val
        normalized = normalized.clip(min_val, max_val)
        
        return normalized.round(2)
    except Exception as e:
        print(f"Error in normalize_metric: {str(e)}")
        return pd.Series([0] * len(series))

In [35]:
def analyze_striker_performance(competition_id, season_id):
    """
    Analyze striker performance across multiple metrics
    Returns a DataFrame with normalized scores
    """
    try:
        matches = sb.matches(competition_id=competition_id, season_id=season_id)
        
        striker_stats = defaultdict(lambda: {
            'minutes_played': 0,
            'finishing': 0,
            'early_shots': 0,
            'linkup_play': 0,
            'carrying': 0,
            'work_rate': 0,
            'back_to_goal': 0,
            'headed_duels': 0,
            'matches_played': 0
        })
        
        striker_positions = [
            'Center Forward',
            'Striker',
            'Second Striker',
            'Centre Forward',
            'Left Center Forward',
            'Right Center Forward',
            'Forward'
        ]
        
        for _, match in matches.iterrows():
            try:
                events = sb.events(match_id=match['match_id'])
                
                if 'position' not in events.columns:
                    continue
                
                strikers = events[
                    events['position'].isin(striker_positions)
                ]['player'].unique()
                
                for striker in strikers:
                    striker_events = events[events['player'] == striker]
                    if len(striker_events) == 0:
                        continue
                    
                    team = striker_events['team'].iloc[0]
                    minutes = striker_events['minute'].max() - striker_events['minute'].min()
                    
                    stats = striker_stats[striker]
                    stats['minutes_played'] += minutes
                    stats['finishing'] += calculate_finishing(events, striker, team)
                    stats['early_shots'] += calculate_early_shots(events, striker, team)
                    stats['linkup_play'] += calculate_linkup_play(events, striker, team)
                    stats['carrying'] += calculate_carrying(events, striker, team)
                    stats['work_rate'] += calculate_work_rate(events, striker, team)
                    stats['back_to_goal'] += calculate_back_to_goal(events, striker, team)
                    stats['headed_duels'] += calculate_headed_duels(events, striker, team)
                    stats['matches_played'] += 1
                    
            except Exception:
                continue
        
        df = pd.DataFrame.from_dict(striker_stats, orient='index')
        if len(df) == 0:
            return pd.DataFrame()
        
        metrics = ['finishing', 'early_shots', 'linkup_play', 'carrying', 
                  'work_rate', 'back_to_goal', 'headed_duels']
        
        df['matches_played'] = df['matches_played'].replace(0, 1)
        
        for metric in metrics:
            df[metric] = df[metric] / df['matches_played']
        
        for metric in metrics:
            df[metric] = normalize_metric(
                df[metric],
                minutes_played=df['minutes_played']
            )
        
        df['avg_minutes_per_match'] = df['minutes_played'] / df['matches_played']
        
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        df[numeric_columns] = df[numeric_columns].round(2)
        
        return df
        
    except Exception:
        return pd.DataFrame()

def analyze_multiple_leagues(leagues):
    """
    Analyze striker performance across multiple leagues and seasons
    Returns a consolidated DataFrame with one entry per player (highest minutes played)
    """
    all_data = []
    
    for league in leagues:
        competition_id = league["competition_id"]
        for season_id in league["season_ids"]:
            try:
                print(f"Analyzing competition {competition_id}, season {season_id}")
                striker_analysis = analyze_striker_performance(competition_id, season_id)
                
                if len(striker_analysis) > 0:
                    striker_analysis['competition_id'] = competition_id
                    striker_analysis['season_id'] = season_id
                    all_data.append(striker_analysis)
                    
            except Exception:
                continue
    
    if not all_data:
        return pd.DataFrame()
    
    combined_df = pd.concat(all_data, axis=0)
    combined_df = combined_df.reset_index().rename(columns={'index': 'player'})
    
    deduplicated_df = (combined_df.sort_values('minutes_played', ascending=False)
                      .groupby('player').first()
                      .reset_index())
    
    final_df = deduplicated_df.sort_values('finishing', ascending=False)
    final_df.to_csv('consolidated_striker_analysis.csv', index=False)
    
    print(f"\nAnalysis complete. Found {len(final_df)} unique strikers.")
    print("Results saved to consolidated_striker_analysis.csv")
    
    return final_df

# Define leagues to analyze
leagues = [
    {"competition_id": 9, "season_ids": [281]},
    {"competition_id": 43, "season_ids": [106]},
    {"competition_id": 11, "season_ids": [90, 42, 4]},
    {"competition_id": 7, "season_ids": [235, 108]},
    {"competition_id": 2, "season_ids": [44]},
    {"competition_id": 12, "season_ids": [27]},
    {"competition_id": 55, "season_ids": [282]},
]

# Run the analysis
try:
    result_df = analyze_multiple_leagues(leagues)
    if len(result_df) > 0:
        print("\nTop 10 strikers by finishing score:")
        print(result_df[['player', 'finishing', 'minutes_played', 'matches_played', 'competition_id', 'season_id']].head(10))
except Exception:
    pass

Analyzing competition 9, season 281
Analyzing competition 43, season 106
Analyzing competition 11, season 90
Analyzing competition 11, season 42
Analyzing competition 11, season 4
Analyzing competition 7, season 235
Analyzing competition 7, season 108
Analyzing competition 2, season 44
Analyzing competition 12, season 27
Analyzing competition 55, season 282

Analysis complete. Found 713 unique strikers.
Results saved to consolidated_striker_analysis.csv

Top 10 strikers by finishing score:
                             player  finishing  minutes_played  \
655                Teddy Sheringham      10.00              91   
142            Daniel Olmo Carvajal      10.00              93   
231              Georges Mikautadze      10.00             350   
244         Gonzalo Gerardo Higuaín      10.00            2999   
409                    Ludovic Blas      10.00              93   
527                Omar El Kaddouri       9.94               4   
489                   Mohamed Salah       9